# 📖 Notebook 4: Feed Caching Strategies

Even with precomputed feeds, reading from PostgreSQL for every request won't scale to billions of users.  
This notebook explores how to use **Redis** to cache feeds and handle the trickiest problem: **hot keys**.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to cache precomputed feeds in Redis sorted sets
- The difference between sharded and replicated caches
- How to handle cache invalidation when new posts arrive
- The hot key problem and how to solve it with replicated caches

## 🛠️ Setup

```bash
cd 06-system-designs/fb-news-feed
docker compose up -d
```

### Visualization Tools

- **RedisInsight**: http://localhost:5540 — watch keys appear and expire in real time  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import random

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "newsfeed_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Why Cache the Feed?

Even with a `precomputed_feed` table and indexes, every feed read still:
1. Opens a database connection
2. Executes a query (index scan + JOIN)
3. Transfers rows over the network

At Facebook's scale (2B users checking feeds constantly), this overwhelms PostgreSQL.  
**Redis** can serve a cached feed in ~1 ms vs ~5-10 ms from PostgreSQL.

### Strategy: Redis Sorted Sets

We'll store each user's feed as a **sorted set** in Redis:
- **Key**: `feed:{user_id}`
- **Members**: JSON-encoded post data
- **Score**: post timestamp (for ordering)

This gives us O(log N) insertions and O(log N + K) range queries (K = page size).

In [ ]:
FEED_CACHE_TTL = 300  # 5 minutes
FEED_MAX_SIZE = 200   # keep only the 200 most recent posts per feed


def cache_feed_for_user(user_id: int, r: redis.Redis) -> int:
    """
    Load a user's feed from PostgreSQL into a Redis sorted set.
    Score = Unix timestamp so newest posts have the highest score.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    cur.execute("""
        SELECT pf.post_id, p.content, pf.post_created_at,
               u.username, u.display_name
        FROM precomputed_feed pf
        JOIN posts p ON p.id = pf.post_id
        JOIN users u ON u.id = pf.post_author_id
        WHERE pf.user_id = %s
        ORDER BY pf.post_created_at DESC
        LIMIT %s
    """, (user_id, FEED_MAX_SIZE))
    posts = cur.fetchall()
    conn.close()
    
    if not posts:
        return 0
    
    key = f"feed:{user_id}"
    pipe = r.pipeline()
    pipe.delete(key)
    
    for post in posts:
        member = json.dumps({
            "post_id": post["post_id"],
            "content": post["content"],
            "username": post["username"],
            "display_name": post["display_name"],
            "created_at": post["post_created_at"].isoformat()
        })
        score = post["post_created_at"].timestamp()
        pipe.zadd(key, {member: score})
    
    pipe.expire(key, FEED_CACHE_TTL)
    pipe.execute()
    
    return len(posts)


def read_feed_from_cache(user_id: int, r: redis.Redis, offset: int = 0, limit: int = 10) -> list:
    """
    Read a page of the user's feed from Redis.
    Returns posts from newest to oldest.
    """
    key = f"feed:{user_id}"
    raw = r.zrevrange(key, offset, offset + limit - 1)
    return [json.loads(item) for item in raw]


# Cache feed for user 1
r = get_redis()
count = cache_feed_for_user(1, r)
print(f"📥 Cached {count} posts for user 1")

# Read back from cache
cached_feed = read_feed_from_cache(1, r, offset=0, limit=5)
print(f"\n📰 User 1's cached feed (first 5):")
for post in cached_feed:
    print(f"  [{post['created_at'][:16]}] @{post['username']}: {post['content'][:45]}")

print(f"\n⏰ TTL: {r.ttl('feed:1')} seconds")

## Cache vs Database: Speed Comparison

In [ ]:
def read_feed_from_db(user_id: int, limit: int = 10) -> list:
    """Read feed directly from PostgreSQL (no cache)."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT pf.post_id, p.content, pf.post_created_at,
               u.username, u.display_name
        FROM precomputed_feed pf
        JOIN posts p ON p.id = pf.post_id
        JOIN users u ON u.id = pf.post_author_id
        WHERE pf.user_id = %s
        ORDER BY pf.post_created_at DESC
        LIMIT %s
    """, (user_id, limit))
    result = cur.fetchall()
    conn.close()
    return result


# Make sure feed is cached
r = get_redis()
cache_feed_for_user(1, r)

# Benchmark: PostgreSQL vs Redis
iterations = 100

# PostgreSQL
times_db = []
for _ in range(iterations):
    t0 = time.time()
    read_feed_from_db(1, limit=20)
    times_db.append((time.time() - t0) * 1000)

# Redis
times_cache = []
for _ in range(iterations):
    t0 = time.time()
    read_feed_from_cache(1, r, limit=20)
    times_cache.append((time.time() - t0) * 1000)

avg_db = sum(times_db) / len(times_db)
avg_cache = sum(times_cache) / len(times_cache)

print(f"⏱️  Feed Read Latency ({iterations} iterations):")
print(f"   PostgreSQL: {avg_db:.2f} ms (avg), {sorted(times_db)[95]:.2f} ms (p95)")
print(f"   Redis:      {avg_cache:.2f} ms (avg), {sorted(times_cache)[95]:.2f} ms (p95)")
print(f"   Speedup:    {avg_db / avg_cache:.1f}×")
print()
print("💡 Redis is significantly faster for reads — this is why we cache feeds!")

## Cache Invalidation: Pushing New Posts

When someone creates a new post, we need to update the cached feeds of their followers.  
We **don't** need to rebuild the entire feed — just push the new post into the sorted set.

In [ ]:
def push_post_to_cached_feeds(author_id: int, post_id: int, content: str,
                               created_at: str, r: redis.Redis):
    """
    When a new post is created, push it into each follower's cached feed.
    Only updates followers who have an active cache (no point updating expired caches).
    """
    conn = get_db()
    cur = conn.cursor()
    
    # Get author info
    cur.execute("SELECT username, display_name FROM users WHERE id = %s", (author_id,))
    username, display_name = cur.fetchone()
    
    # Get followers
    cur.execute("SELECT follower_id FROM follows WHERE followee_id = %s", (author_id,))
    follower_ids = [row[0] for row in cur.fetchall()]
    conn.close()
    
    member = json.dumps({
        "post_id": post_id,
        "content": content,
        "username": username,
        "display_name": display_name,
        "created_at": created_at
    })
    
    import datetime
    score = datetime.datetime.fromisoformat(created_at).timestamp()
    
    updated = 0
    pipe = r.pipeline()
    for fid in follower_ids:
        key = f"feed:{fid}"
        if r.exists(key):  # only update active caches
            pipe.zadd(key, {member: score})
            # Trim to keep only the most recent FEED_MAX_SIZE posts
            pipe.zremrangebyrank(key, 0, -(FEED_MAX_SIZE + 1))
            updated += 1
    pipe.execute()
    
    return updated


# Demo: cache feeds for a few users, then create a post
r = get_redis()
for uid in [1, 2, 3, 4, 5]:
    cache_feed_for_user(uid, r)

# User 1's feed before the new post
before = read_feed_from_cache(1, r, limit=3)
print("📰 User 1's feed BEFORE new post:")
for p in before:
    print(f"   @{p['username']}: {p['content'][:50]}")

# Celebrity Alice (id=51) creates a new post
conn = get_db()
cur = conn.cursor()
cur.execute(
    "INSERT INTO posts (author_id, content) VALUES (%s, %s) RETURNING id, created_at",
    (51, "🔥 Breaking news from Celebrity Alice! Big announcement!")
)
new_post_id, new_created_at = cur.fetchone()
conn.commit()
conn.close()

# Push to cached feeds
updated = push_post_to_cached_feeds(
    author_id=51,
    post_id=new_post_id,
    content="🔥 Breaking news from Celebrity Alice! Big announcement!",
    created_at=new_created_at.isoformat(),
    r=r
)
print(f"\n📤 Pushed to {updated} cached feeds")

# User 1's feed after the new post
after = read_feed_from_cache(1, r, limit=3)
print("\n📰 User 1's feed AFTER new post:")
for p in after:
    print(f"   @{p['username']}: {p['content'][:50]}")

print("\n💡 The new celebrity post appears at the top of the feed instantly!")

## 🔥 The Hot Key Problem

In a **sharded** cache (Redis Cluster), each key lives on exactly one node.  
When a celebrity post goes viral, the node holding that post's cache gets hammered.

```
Sharded Cache (PROBLEM):

   Node A          Node B          Node C
┌──────────┐   ┌──────────┐   ┌──────────┐
│ post:1   │   │ post:2   │   │ post:3   │
│ post:4   │   │ post:5   │   │ post:6   │
│ VIRAL!!! │   │          │   │          │
│ 1M req/s │   │ 100 req/s│   │ 100 req/s│
└──────────┘   └──────────┘   └──────────┘
   ❌ OVERLOADED     Idle          Idle
```

### Solution: Replicated Caches

Instead of sharding by key, **replicate** the cache so every node can serve any key.
A load balancer spreads requests evenly across all nodes.

```
Replicated Cache (SOLUTION):

   Node A          Node B          Node C
┌──────────┐   ┌──────────┐   ┌──────────┐
│ ALL posts│   │ ALL posts│   │ ALL posts│
│ VIRAL!!! │   │ VIRAL!!! │   │ VIRAL!!! │
│ 333K r/s │   │ 333K r/s │   │ 333K r/s │
└──────────┘   └──────────┘   └──────────┘
   ✅ Balanced    ✅ Balanced    ✅ Balanced
```

### What Replication Actually Costs

Replication is not free, and "just replicate the cache" is a weak interview
answer unless you say the price out loud:

| Cost | Detail |
|---|---|
| **Memory ×N** | Every node holds the *entire* dataset. Three nodes means three copies. Sharding exists so your working set can exceed one machine's RAM — replication throws that away. |
| **Write amplification ×N** | Every `SET` and every invalidation must reach every node, so writes get N× more expensive exactly as reads get N× cheaper. |
| **Inconsistency window** | Nodes diverge briefly during a write. Two users refreshing at the same instant can see different feeds. |

**Cheaper alternatives worth naming, in the order you'd reach for them:**

1. **In-process cache with a 1-second TTL** on each app server. A viral key gets
   read from local memory; Redis sees at most one request per server per second.
   Costs you up to 1s of staleness and nothing else. This handles most hot keys.
2. **Request coalescing / single-flight** — when 10,000 concurrent requests miss
   the same key, let one of them fetch and make the rest wait on that result.
   Turns a thundering herd into a single query.
3. **Key splitting** — store the hot value under `post:1:copy0..copy9` and have
   clients pick one at random. Spreads one hot key over 10 shards without
   replicating the other 99.99% of the keyspace.

Replicate only what is *provably* hot, and only after 1 and 2 are in place.

In [ ]:
# Simulate the hot key problem with sharded vs replicated caches

NUM_NODES = 3
TOTAL_REQUESTS = 10000

# Simulate posts with varying popularity
# post_id -> number of requests it receives
post_popularity = {
    1: 7000,  # viral post!
    2: 1000,
    3: 500,
    4: 500,
    5: 400,
    6: 300,
    7: 200,
    8: 100,
}

# Sharded: each post lives on exactly one node (hash-based assignment)
sharded_load = [0] * NUM_NODES
for post_id, reqs in post_popularity.items():
    node = hash(post_id) % NUM_NODES
    sharded_load[node] += reqs

# Replicated: requests are round-robin across all nodes
replicated_load = [0] * NUM_NODES
all_requests = []
for post_id, reqs in post_popularity.items():
    all_requests.extend([post_id] * reqs)
random.shuffle(all_requests)

for i, req in enumerate(all_requests):
    node = i % NUM_NODES  # round-robin load balancer
    replicated_load[node] += 1

print("🔥 Hot Key Problem Simulation")
print("=" * 50)
print(f"   Total requests: {TOTAL_REQUESTS}")
print(f"   Post 1 (viral): {post_popularity[1]} requests ({post_popularity[1]/TOTAL_REQUESTS*100:.0f}%)")
print()

print("📊 Sharded Cache (post pinned to one node):")
for i, load in enumerate(sharded_load):
    bar = '█' * (load // 200)
    print(f"   Node {i}: {load:>6} requests  {bar}")
print(f"   Max/Min ratio: {max(sharded_load) / max(min(sharded_load), 1):.1f}×")
print()

print("📊 Replicated Cache (load balanced):")
for i, load in enumerate(replicated_load):
    bar = '█' * (load // 200)
    print(f"   Node {i}: {load:>6} requests  {bar}")
print(f"   Max/Min ratio: {max(replicated_load) / max(min(replicated_load), 1):.1f}×")
print()
print("💡 Replicated caches distribute hot key traffic evenly across all nodes!")

## 📄 Pagination: Offset vs Keyset (Cursor)

When a user scrolls down, we fetch the next page. There are two ways:

| Approach | How it works | Problem |
|---|---|---|
| ❌ **Offset pagination** | `LIMIT 10 OFFSET 90` — skip 90 rows | As offset grows, the DB still scans 100 rows to return 10 — **O(offset)** cost. And if a new post arrives between pages, rows shift, so users see duplicates or skipped posts. |
| ✅ **Keyset / cursor pagination** | `WHERE created_at < last_seen_ts LIMIT 10` | The cursor is the **score** of the last post you saw. New posts at the top never shift older pages. **O(log N)** cost regardless of depth. |

Redis sorted sets support keyset pagination natively via `ZREVRANGEBYSCORE` with an exclusive upper bound.


In [ ]:
def read_feed_page_keyset(user_id, r, cursor=None, limit=5):
    '''
    Keyset pagination using a Redis sorted set.
    `cursor` is the timestamp of the last post the client has seen.
    Pass None for the first page. Returns (posts, next_cursor).
    '''
    key = f"feed:{user_id}"
    # "(value" means EXCLUSIVE upper bound in Redis — we skip the last post
    max_score = "+inf" if cursor is None else f"({cursor}"
    raw = r.zrevrangebyscore(key, max_score, "-inf", start=0, num=limit, withscores=True)
    posts = [json.loads(item) for item, _score in raw]
    next_cursor = raw[-1][1] if raw else None
    return posts, next_cursor


# Rebuild cache for user 1 and walk through three pages
r = get_redis()
cache_feed_for_user(1, r)

cursor = None
for page_num in range(1, 4):
    posts, cursor = read_feed_page_keyset(1, r, cursor=cursor, limit=3)
    if not posts:
        break
    print(f"📄 Page {page_num} (next cursor → {cursor}):")
    for p in posts:
        print(f"   [{p['created_at'][:16]}] @{p['username']}: {p['content'][:40]}")
    print()

print("💡 Each page starts STRICTLY below the previous cursor, so new posts")
print("   arriving at the top of the feed never push old rows into the next page.")
print("   Offset pagination ('LIMIT 10 OFFSET 90') cannot make that guarantee.")


## Cache-Aside with Fallback

In production, you use a **cache-aside** pattern with a database fallback:  
1. Check Redis for the feed  
2. If cache **hit** → return it  
3. If cache **miss** → read from PostgreSQL → populate cache → return

In [ ]:
def get_feed(user_id: int, offset: int = 0, limit: int = 10) -> dict:
    """
    Production-style feed read with cache-aside pattern.
    """
    r = get_redis()
    key = f"feed:{user_id}"
    
    # Step 1: Try cache
    if r.exists(key):
        feed = read_feed_from_cache(user_id, r, offset, limit)
        return {"source": "cache", "posts": feed}
    
    # Step 2: Cache miss — read from DB
    feed_db = read_feed_from_db(user_id, limit=FEED_MAX_SIZE)
    
    # Step 3: Populate cache for next time
    cache_feed_for_user(user_id, r)
    
    # Step 4: Return the requested page
    page = feed_db[offset:offset + limit]
    return {"source": "database", "posts": page}


# First call: cache miss (loads from DB)
# Clear any existing cache first
r = get_redis()
r.delete("feed:10")

result1 = get_feed(10, limit=3)
print(f"1st call: source={result1['source']}, posts={len(result1['posts'])}")

# Second call: cache hit!
result2 = get_feed(10, limit=3)
print(f"2nd call: source={result2['source']}, posts={len(result2['posts'])}")

# Latency comparison
r.delete("feed:20")

t0 = time.time()
get_feed(20)  # miss
miss_time = (time.time() - t0) * 1000

t0 = time.time()
get_feed(20)  # hit
hit_time = (time.time() - t0) * 1000

print(f"\n⏱️  Cache miss: {miss_time:.1f} ms (DB read + cache write)")
print(f"⏱️  Cache hit:  {hit_time:.1f} ms")
print(f"💡 After the first read, all subsequent reads are served from cache!")

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys
r = get_redis()
keys = r.keys("feed:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} feed cache keys")

# Clean up test posts
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM precomputed_feed WHERE post_id IN (SELECT id FROM posts WHERE content LIKE '%Breaking news from Celebrity%')")
cur.execute("DELETE FROM posts WHERE content LIKE '%Breaking news from Celebrity%'")
conn.commit()
conn.close()
print("🧹 Cleaned up test posts")

## 📚 Summary

### Key Takeaways

1. **Redis sorted sets** are perfect for feed caching — ordered by timestamp, O(log N) operations
2. **Cache-aside** pattern: try cache first, fall back to DB, then populate cache
3. **Push invalidation**: when a new post is created, push it directly into cached feeds
4. **TTL** ensures stale caches eventually expire (5 min is typical for feeds)
5. **Replicated caches** solve the hot key problem by distributing viral post traffic

### Interview Tips

- Mention that feeds are cached in Redis sorted sets — interviewers love specifics
- Explain the hot key problem before the interviewer brings it up
- The sharded vs replicated cache trade-off is a great deep-dive topic
- Note that replicated caches use more memory but handle hot keys gracefully
- For post content, use a separate cache (cache-aside with LRU) keyed by post ID

### Series Complete! 🎉

You've now covered the four pillars of a News Feed system:
1. **Fan-out strategies** — how to build and deliver feeds
2. **Ranking** — how to order posts by relevance
3. **Social graph** — how to store and query relationships
4. **Caching** — how to serve feeds at massive scale